# Agent Hypervisor — Compiler Pipeline Walkthrough

This notebook walks through the four steps of the **design-time compiler pipeline**:

```
Compiler Step 0   Compiler Step 1   Compiler Step 2   Compiler Step 3   Compiler Step 4
   Record       →    Observe       →    Profile       →    Compile       →    Enforce
(ExecutionTrace)   (inspect)          (capabilities)   (WorldManifest)    (EvalResult)
```

> **Naming note.** These are *compiler pipeline steps* — design-time stages that run
> *before the agent exists*. They are distinct from the architectural execution stages
> (Stage 0–6) described in `architecture.md`, which describe runtime execution flow.

The notebook also covers the runtime enforcement layer:
- How `TaintContext` threads taint monotonically between actions
- The **anti-laundering invariant**: combining a tainted value with a clean value yields a tainted result

## Setup

In [ ]:
import json
import sys
from pathlib import Path

import yaml

# Locate the repo root (directory that contains pyproject.toml)
_cwd = Path('.').resolve()
REPO_ROOT = _cwd
for _ in range(6):
    if (REPO_ROOT / 'pyproject.toml').exists():
        break
    REPO_ROOT = REPO_ROOT.parent
else:
    raise RuntimeError('Could not find repo root — run this notebook from inside the repository')

# Make agent_hypervisor importable without pip install
_src = str(REPO_ROOT / 'src')
if _src not in sys.path:
    sys.path.insert(0, _src)

# Fixture traces shipped with the compiler module
FIXTURES = REPO_ROOT / 'src' / 'agent_hypervisor' / 'compiler' / 'fixtures'

print(f'Repo root : {REPO_ROOT}')
print(f'Fixtures  : {FIXTURES}')

---
## Compiler Step 0 — Record

Before a trace can be profiled it must be **produced**. At design time, a recorder
wraps the agent's tool calls and appends each invocation to an `ExecutionTrace`. The
resulting JSON file is the raw material for all later steps.

There is no `TraceRecorder` class in the current API — the data model itself is the
interface. Below we construct a trace inline to show the structure:

In [ ]:
from agent_hypervisor.compiler.observe import ExecutionTrace, ToolCall

inline_trace = ExecutionTrace(
    workflow_id='repo-maintenance',
    calls=[
        ToolCall(tool='read_file',  params={'path': 'docs/index.md'},        safe=True),
        ToolCall(tool='read_file',  params={'path': 'docs/api.md'},          safe=True),
        ToolCall(tool='web_search', params={'domain': 'docs.python.org'},    safe=True),
        ToolCall(tool='write_file', params={'path': 'output/summary.md'},    safe=True),
    ],
    metadata={'description': 'Benign docs-summarisation workflow, recorded inline.'},
)

print(f'workflow_id : {inline_trace.workflow_id}')
print(f'calls       : {len(inline_trace.calls)}')
for call in inline_trace.calls:
    flag = '✅' if call.safe else '❌'
    print(f'  {flag} {call.tool:<18} params={call.params}')

> `safe=True` is the annotation that controls whether a call contributes to the
> capability profile. It is set by the operator reviewing the trace — not by the
> agent. An operator marking a call `safe=False` is the canonical way to exclude
> a step from ever widening the capability surface.

---
## Compiler Step 1 — Observe

Load recorded traces from JSON files and inspect their structure.

### Benign trace

In [ ]:
from agent_hypervisor.compiler.observe import load_trace

benign_trace = load_trace(FIXTURES / 'benign_trace.json')

print(f'workflow_id : {benign_trace.workflow_id}')
print(f'calls       : {len(benign_trace.calls)}')
for call in benign_trace.calls:
    flag = '✅' if call.safe else '❌'
    print(f'  {flag} {call.tool:<18} params={call.params}')

### Unsafe trace — mixed safe and unsafe calls

In [ ]:
unsafe_trace = load_trace(FIXTURES / 'unsafe_trace.json')

print(f'workflow_id : {unsafe_trace.workflow_id}')
print(f'calls       : {len(unsafe_trace.calls)}')
for call in unsafe_trace.calls:
    flag = '✅' if call.safe else '❌'
    print(f'  {flag} {call.tool:<18} params={call.params}')

> The unsafe trace contains both `safe=True` and `safe=False` calls.
> The profiler in Step 2 will include only the `safe=True` ones.

---
## Compiler Step 2 — Profile

The profiler reduces one or more traces into a **capability profile** — the minimal
set of `(tool, constraints)` pairs observed across all `safe=True` calls.

**Safe compression invariant:** a tool, path, or domain not observed in a `safe=True`
call cannot appear in the profile. Capability can only shrink; it cannot expand.

In [ ]:
from agent_hypervisor.compiler.profile import profile_trace

benign_profile = profile_trace(benign_trace)

print('Capability profile — benign trace (all calls safe):',)
for cap in benign_profile:
    print(f'  tool={cap.tool!r:<22} constraints={cap.constraints}')

In [ ]:
unsafe_profile = profile_trace(unsafe_trace)

print('Capability profile — unsafe trace (only safe=True calls contribute):')
for cap in unsafe_profile:
    print(f'  tool={cap.tool!r:<22} constraints={cap.constraints}')

print()
# exec_shell and the unsafe web_search domain are absent — they came from safe=False calls
benign_tools = {c.tool for c in benign_profile}
unsafe_tools  = {c.tool for c in unsafe_profile}
print(f'Benign profile tools : {benign_tools}')
print(f'Unsafe profile tools : {unsafe_tools}')
print(f'Excluded from unsafe : {benign_tools - unsafe_tools} (unsafe calls excluded)')

---
## Compiler Step 3 — Compile

The compiler wraps the capability profile in a `WorldManifest` — the declarative
boundary document that the enforcement engine consumes at runtime.

The manifest format:
```yaml
workflow_id: ...
version: '1.0'
capabilities:
  - tool: read_file
    constraints:
      paths: ['docs/index.md', 'docs/api.md']
  - tool: web_search
    constraints:
      domains: ['docs.python.org']
```

In [ ]:
from agent_hypervisor.compiler.profile import build_manifest
from agent_hypervisor.compiler.schema import manifest_to_dict

manifest = build_manifest(benign_trace)

print(f'WorldManifest for: {manifest.workflow_id!r}')
print(f'Permitted tools  : {manifest.tool_names()}')
print()
print(yaml.dump(manifest_to_dict(manifest), default_flow_style=False, sort_keys=False))

> `http_post`, `exec_shell`, and any tool not in this manifest do not exist in this
> agent's world. They cannot be invoked — not because a filter will block them, but
> because they have no representation. This is the difference between `DENY_ABSENT`
> and `DENY_POLICY`, which Step 4 demonstrates.

---
## Compiler Step 4 — Enforce

The enforcement engine evaluates a `Step` against a `WorldManifest` and returns one
of three **deterministic** decisions:

| Decision | Meaning |
| --- | --- |
| `ALLOW` | Step is within the declared capability profile |
| `DENY_ABSENT` | Tool has no representation in this manifest — ontological absence |
| `DENY_POLICY` | Tool exists but this call violates a constraint (taint, path, remote) |

Both denial types are deterministic. Neither involves judgment or probability.

> **`Step` vs `ToolCall`:** `ToolCall` (Step 0–2) is the design-time observation.
> `Step` (Step 4) is the runtime proposal. They carry different fields — `ToolCall`
> has `safe=`; `Step` has `action`, `resource`, and `input_sources` for taint tracking.

In [ ]:
from agent_hypervisor.compiler.enforcer import Step, Decision, evaluate

icons = {
    Decision.ALLOW:            '✅',
    Decision.DENY_ABSENT:      '🚫',
    Decision.DENY_POLICY:      '⛔',
    Decision.REQUIRE_APPROVAL: '⚠️',
}

steps = [
    # ALLOW — tool and path both in manifest
    Step(tool='read_file',  action='read', resource='docs/index.md',
         input_sources=['repo_local']),
    # DENY_POLICY — tool exists, but path not in allowlist
    Step(tool='read_file',  action='read', resource='/etc/passwd',
         input_sources=['repo_local']),
    # DENY_ABSENT — tool has no manifest entry at all
    Step(tool='http_post',  action='post', resource='https://attacker.com/exfil',
         input_sources=['repo_local']),
    # DENY_POLICY — tainted input; 'tainted' in input_sources triggers policy violation
    Step(tool='read_file',  action='read', resource='docs/index.md',
         input_sources=['tainted']),
]

print(f'Manifest: {manifest.workflow_id!r}  |  permitted tools: {manifest.tool_names()}')
print()
for step in steps:
    result = evaluate(step, manifest)
    icon = icons[result.decision]
    print(f'{icon} {result.decision.value:<20} {step.display_name:<18}  resource={step.resource!r}')
    print(f'   reason: {result.reason}')
    print()

Notice the two distinct denial types:

- `DENY_ABSENT` (`http_post`) — the tool simply does not exist in this world.
  There is no entry to match against. Evaluation never begins.
- `DENY_POLICY` (`read_file /etc/passwd`, `read_file` with tainted input) — the tool
  exists, but *this specific call* violates a constraint.

Both are immune to rephrasing: the enforcement boundary operates below the language level.

### Interactive — build a custom step

In [ ]:
# Modify this step and re-run the cell to explore the enforcement engine.
# Ideas:
#   - change resource to 'docs/api.md' → ALLOW
#   - change tool to 'write_file', resource to 'output/summary.md' → ALLOW
#   - add 'tainted' to input_sources → DENY_POLICY even for an allowed path

custom = Step(
    tool='web_search',
    action='search',
    resource='malicious.example.com',   # not in the domains allowlist
    input_sources=['repo_local'],
)

result = evaluate(custom, manifest)
print(f'{icons[result.decision]} {result.decision.value}  —  {result.reason}')
print(f'failure_type: {result.failure_type}')

---
## Runtime Enforcement Layer — TaintContext and TaintedValue

The compiler-layer taint above (`'tainted' in step.input_sources`) is a simplified
signal for the enforcer. The full runtime uses a structural taint system:

- **`TaintedValue[T]`** — every value returned by `Executor.execute()` is wrapped
  in a `TaintedValue`. The taint state is part of the value; it cannot be removed.
- **`TaintContext`** — threads taint between pipeline steps. Passed *explicitly* to
  `IRBuilder.build()` — it is not optional, so taint can never be dropped by omission.
- **`TaintState.CLEAN / TAINTED`** — the two states. Join is monotonic:
  `CLEAN ∨ TAINTED = TAINTED`.

In [ ]:
from agent_hypervisor.runtime import TaintContext, TaintedValue, TaintState

# A clean value from a trusted source
clean_val = TaintedValue(value='pytest output', taint=TaintState.CLEAN)
# A tainted value from an external document
tainted_val = TaintedValue(value='attacker@evil.com', taint=TaintState.TAINTED)

print(f'clean_val   : {clean_val}')
print(f'tainted_val : {tainted_val}')
print()

# .map() transforms the value while preserving taint — wrapping doesn't launder
transformed = tainted_val.map(str.upper)
print(f'tainted_val.map(str.upper) : {transformed}')
print('Taint survives the transformation. The value changed; the taint did not.')

### Anti-laundering invariant

A derived value inherits the **least-trusted** taint among all its parents.
Combining a clean value with a tainted value always yields a tainted result.
There is no operation that reduces taint.

In [ ]:
# Scenario: agent reads a safe summary (clean) and an external email (tainted),
# then tries to combine them into a send_email call.

clean_summary  = TaintedValue(value='Q1 results summary', taint=TaintState.CLEAN)
malicious_addr = TaintedValue(value='exfil@attacker.com', taint=TaintState.TAINTED)

# Monotonic join: any TAINTED input dominates
combined_taint = TaintedValue.join(clean_summary, malicious_addr)

print(f'clean_summary.taint   : {clean_summary.taint}')
print(f'malicious_addr.taint  : {malicious_addr.taint}')
print(f'TaintedValue.join()   : {combined_taint}  ← external_document dominates')
print()
print('Even though one input is clean, the combination is TAINTED.')
print('Constructing a new TaintedValue that labels this as clean would not help:')
print('IRBuilder.build() receives the TaintContext, which carries the join result.')
print()

# TaintContext propagates the joined taint into the next IRBuilder.build() call
ctx_clean = TaintContext.clean()
ctx_mixed = TaintContext.from_outputs(clean_summary, malicious_addr)

print(f'TaintContext.clean()                          → {ctx_clean}')
print(f'TaintContext.from_outputs(clean, tainted)     → {ctx_mixed}')

> **Why this matters.** The zombie attack scenario chains only legitimate actions,
> but the data flows from an untrusted source through to an external call:
>
> ```
> read_file (untrusted doc)   →  ALLOW
> summarize                   →  ALLOW
> send_email (external)       →  DENY_POLICY  ← taint propagated from the read
> ```
>
> Taint propagation catches this without the system needing to understand the
> attacker's intent. Each step is individually legal; the chain is not.

### IRBuilder — the enforcement boundary

At runtime, `IRBuilder.build()` is the single point where all constraints are checked
*before any execution begins*. It receives:
- the proposed action and parameters
- a `TaintContext` carrying the monotonic join of all prior outputs

If `build()` returns an `IntentIR`, every constraint has passed. If it raises
`ConstructionError`, nothing executes. The `IntentIR` is sealed — it cannot be
constructed outside `IRBuilder`, so no external code can inject an intent that has
not passed all checks.

---
## Full pipeline — end to end

Record → Profile → Compile → Enforce in a single cell:

In [ ]:
from agent_hypervisor.compiler.observe import ExecutionTrace, ToolCall
from agent_hypervisor.compiler.profile import build_manifest
from agent_hypervisor.compiler.enforcer import Step, Decision, evaluate

# Compiler Step 0: Record
trace = ExecutionTrace(
    workflow_id='ci-pipeline',
    calls=[
        ToolCall(tool='read_file',  params={'path': 'src/**/*.py'},    safe=True),
        ToolCall(tool='web_search', params={'domain': 'pypi.org'},     safe=True),
        ToolCall(tool='write_file', params={'path': 'output/report'}, safe=True),
        # exec_shell marked unsafe — will NOT enter the manifest
        ToolCall(tool='exec_shell', params={'command': 'rm -rf /'},   safe=False),
    ],
)

# Compiler Steps 1-3: Observe → Profile → Compile
manifest = build_manifest(trace)
print(f'Manifest: {manifest.workflow_id!r}  |  tools: {manifest.tool_names()}')
print()

# Compiler Step 4: Enforce
test_steps = [
    Step(tool='read_file',  action='read', resource='src/main.py',
         input_sources=['repo_local']),          # ALLOW
    Step(tool='web_search', action='search', resource='pypi.org',
         input_sources=['repo_local']),          # ALLOW
    Step(tool='exec_shell', action='exec',  resource='rm -rf /',
         input_sources=['repo_local']),          # DENY_ABSENT (safe=False excluded it)
    Step(tool='http_post',  action='post',  resource='evil.com',
         input_sources=['repo_local']),          # DENY_ABSENT (never observed)
    Step(tool='read_file',  action='read', resource='src/main.py',
         input_sources=['tainted']),             # DENY_POLICY (tainted input)
]

icons = {Decision.ALLOW: '✅', Decision.DENY_ABSENT: '🚫',
         Decision.DENY_POLICY: '⛔', Decision.REQUIRE_APPROVAL: '⚠️'}

for step in test_steps:
    result = evaluate(step, manifest)
    print(f"{icons[result.decision]} {result.decision.value:<20} {step.display_name:<18}  — {result.reason}")

---
## Summary

| Compiler Step | Input | Output | Module |
| --- | --- | --- | --- |
| **0 — Record** | Agent tool calls | `ExecutionTrace` (JSON) | `compiler.observe` |
| **1 — Observe** | JSON file | `ExecutionTrace` object | `compiler.observe.load_trace` |
| **2 — Profile** | `ExecutionTrace` | `list[CapabilityConstraint]` | `compiler.profile.profile_trace` |
| **3 — Compile** | `ExecutionTrace` | `WorldManifest` | `compiler.profile.build_manifest` |
| **4 — Enforce** | `Step` + `WorldManifest` | `EvalResult` | `compiler.enforcer.evaluate` |

Runtime layer (enforcement kernel):

| Component | Role |
| --- | --- |
| `TaintedValue[T]` | Pairs every executor output with its taint state |
| `TaintContext` | Threads monotonic taint join into `IRBuilder.build()` |
| `IRBuilder.build()` | Single enforcement boundary — all constraints checked before execution |
| `IntentIR` | Sealed — only constructable via `IRBuilder`; signals all constraints passed |